In [108]:
import numpy as np
import pandas as pd
import seaborn as sns

In [109]:
df=pd.read_csv('fifa_world_cup_2026_player_performance.csv')

In [110]:
df.head()

,player_id,player_name,age,nationality,team,jersey_number,position,height_cm,weight_kg,preferred_foot,...,possession_impact,pressure_resistance,creativity_score,consistency_score,clutch_performance_score,total_goals_tournament,total_assists_tournament,total_minutes_tournament,player_of_match_awards,tournament_rating
0,P00055,Rodri Fati,26,Spanish,Spain,3,Goalkeeper,195,75,Left,...,1.1,44.2,55.9,42.0,51.8,0,0,242,0,5.8
1,P00070,Ansu Le Normand,19,Spanish,Spain,18,Midfielder,178,75,Right,...,3.5,38.2,43.7,31.1,52.7,0,3,342,0,5.5
2,P00066,Gavi Ramos,18,Spanish,Spain,14,Midfielder,177,72,Left,...,15.3,99.0,99.0,83.4,54.8,1,1,245,0,8.4
3,P00073,Pedro Cubarsi,20,Spanish,Spain,21,Forward,182,74,Right,...,1.2,19.8,42.3,40.9,78.5,5,3,422,0,6.7
4,P00059,Alvaro Oyarzabal,23,Spanish,Spain,7,Defender,191,81,Left,...,6.2,44.1,33.5,60.0,56.6,0,0,440,0,5.7


In [111]:
groups = df.T.groupby(list(df.T)).groups

found_duplicate =False
for group in groups.values():
    if len(group)>1:
        print(list(group))
        found_duplicate=True
if not found_duplicate:
    print("No duplicate columns")

No duplicate columns


In [112]:
X=df.drop("market_value_eur",axis=1)
y=df['market_value_eur']
num_col=X.select_dtypes(include=['int64','float64']).columns

In [113]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

In [114]:
#from sklearn.preprocessing import StandardScaler
#scaler=StandardScaler()
#X_train[num_col]=scaler.fit_transform(X_train[num_col])
#X_test[num_col]=scaler.transform(X_test[num_col])

In [115]:
from sklearn.feature_selection import VarianceThreshold
sel=VarianceThreshold(threshold=0.1)

In [116]:
sel.fit(X_train[num_col])

VarianceThreshold(threshold=0.1)

In [117]:
low_variance=num_col[~sel.get_support()]
low_variance

Index(['goals', 'assists', 'shots_on_target', 'expected_goals_xg',
       'expected_assists_xa', 'pass_accuracy', 'successful_crosses',
       'yellow_cards', 'red_cards', 'save_percentage', 'punches',
       'clean_sheet', 'penalty_saves', 'player_of_match_awards'],
      dtype='object')

In [118]:
columns=X_train[num_col].columns[sel.get_support()]

In [119]:
X_train=X_train.drop(columns=low_variance)
X_test=X_test.drop(columns=low_variance)

In [120]:
X_train.head()

,player_id,player_name,age,nationality,team,jersey_number,position,height_cm,weight_kg,preferred_foot,...,defensive_contribution,possession_impact,pressure_resistance,creativity_score,consistency_score,clutch_performance_score,total_goals_tournament,total_assists_tournament,total_minutes_tournament,tournament_rating
47033,P00079,Harry Bellingham,33,English,England,1,Goalkeeper,190,80,Right,...,99.0,0.0,99.0,24.2,99.0,99.0,0,0,263,0.0
19860,P01158,Youssef Dahmen,19,Tunisian,Tunisia,14,Midfielder,182,76,Right,...,35.7,0.0,48.8,44.0,53.9,49.5,0,1,343,0.0
1203,P00935,Ajdin Souttar,27,Australian,Australia,25,Forward,180,69,Left,...,39.2,7.9,64.6,77.1,81.1,37.0,5,0,339,7.0
38113,P00641,André Gonzalez,27,Peruvian,Peru,17,Midfielder,177,73,Right,...,38.4,0.0,53.5,51.1,39.7,63.4,1,0,254,0.0
30356,P01218,Omar Wahbi,28,Egyptian,Egypt,22,Forward,173,69,Right,...,31.8,0.0,40.7,38.7,41.7,67.3,1,0,119,0.0


In [121]:
corr_matrix=X_train.corr(numeric_only=True)
columns=corr_matrix.columns
columns_to_drop=[]
for i in range(len(columns)):
    for j in range(i+1,len(columns)):
        if abs(corr_matrix.loc[columns[i],columns[j]])>=0.9:
            columns_to_drop.append(columns[j])

In [122]:
columns_to_drop=set(columns_to_drop)

In [123]:
print(len(columns_to_drop))

9


In [124]:
X_train=X_train.drop(columns=columns_to_drop)
X_test=X_test.drop(columns=columns_to_drop)

In [125]:
X_train.shape

(36582, 51)

In [134]:
X_train=X_train.drop('player_id',axis=1)
X_test=X_test.drop('player_id',axis=1)

In [137]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown='ignore')

X_train = encoder.fit_transform(X_train)
X_test = encoder.transform(X_test)

In [141]:
from sklearn.ensemble import RandomForestRegressor
model2 =RandomForestRegressor(n_estimators=200,random_state=42)


In [ ]:
model2.fit(X_train,y_train)

In [22]:
y_pred=model2.predict(X_test)

In [ ]:
print("Training Score =",model2.score(X_train,y_train))
print("Test Score =",model2.score(X_test,y_test))